# CommaSeparatedListOutputParser - 쉼표 구분 리스트로 변환해요

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요.

- `CommaSeparatedListOutputParser`로 LLM 출력을 Python `list`로 변환할 수 있어요
- 스트리밍 모드에서 항목이 완성될 때마다 실시간으로 출력할 수 있어요
- 키워드 추출, 추천 시스템, 브레인스토밍 등 다양한 실무 시나리오에 적용할 수 있어요

## 사전 지식

- **CommaSeparatedList(쉼표 구분 목록)**: `"항목1, 항목2, 항목3"` 형태의 텍스트를 `["항목1", "항목2", "항목3"]`으로 변환하는 가장 단순한 리스트 파싱 방식이에요

## 파이프라인 흐름

```mermaid
flowchart LR
    A["PromptTemplate<br/>(형식 지침 포함)"] -->|"프롬프트 생성"| B["ChatOpenAI<br/>(LLM)"]
    B -->|"'항목1, 항목2, ...' 텍스트 반환"| C["CommaSeparatedList<br/>OutputParser"]
    C -->|"split + strip 처리"| D["Python list<br/>항목1, 항목2, ..."]

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404

    class A input
    class B,C process
    class D output
```

## 언제 사용하면 좋을까요?

- 키워드, 해시태그, 태그를 **추출**할 때
- 추천 아이템 목록을 **생성**할 때
- 태스크나 주제를 **나열**할 때
- Pydantic 스키마 없이 **빠르게 배열 데이터**를 얻고 싶을 때

> **제한 사항**: 중첩 구조(리스트 안의 딕셔너리 등)는 지원하지 않아요. 복잡한 구조가 필요하면 노트북 02의 `JsonOutputParser`를 사용하세요.

In [2]:
from dotenv import load_dotenv

load_dotenv()


True

## 기본 사용법

가장 간단한 형태의 `CommaSeparatedListOutputParser` 사용 예제입니다.


In [3]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

# 1단계: CommaSeparatedListOutputParser 초기화
output_parser = CommaSeparatedListOutputParser()

# 2단계: 형식 지침 확인
format_instructions = output_parser.get_format_instructions()
print("=" * 60)
print("📋 파서가 생성한 형식 지침")
print("=" * 60)
print(format_instructions)
print()

📋 파서가 생성한 형식 지침
Your response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz`



In [9]:
from torch.fx.experimental.proxy_tensor import prim
# ---------------------------------------------------
# 프롬프트 템플릿 설정 및 체인 구성, 실행
# ---------------------------------------------------

# 1단계: 프롬프트 템플릿 설정
# partial_variables: 템플릿 생성 시 일부 변수를 미리 고정
# TODO: PromptTemplate을 생성하세요 (template, input_variables, partial_variables)
prompt = PromptTemplate(
  template="List five {subject}.\n{format_instructions}",
  input_variables=["subject"],
  partial_variables={"format_instructions": format_instructions}
)

# 2단계: 모델 초기화
model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 3단계: 체인 구성
# CommaSeparatedListOutputParser: "항목1, 항목2, 항목3" → ["항목1", "항목2", "항목3"]
# TODO: prompt | model | output_parser 로 체인을 구성하세요
chain = prompt | model | output_parser

# 4단계: 실행
# TODO: chain.invoke()를 사용하여 실행하세요
res = chain.invoke({"subject": "한국의 유명한 산"})

print("=" * 60)
print("🏔️ 결과 (Python 리스트)")
print("=" * 60)
# TODO: 결과의 타입, 항목 수, 목록을 출력하세요

res


🏔️ 결과 (Python 리스트)


['한라산', '설악산', '지리산', '태백산', '덕유산']

---

## 스트리밍: 각 항목이 완성될 때마다 출력해요

`CommaSeparatedListOutputParser`는 스트리밍을 지원해요. `stream()` 사용 시 쉼표로 항목이 구분될 때마다 새 항목을 즉시 반환해요.

> **실무 팁**: 생성 목록이 길 때 스트리밍을 쓰면 사용자가 기다리는 동안 이미 완성된 항목부터 화면에 표시할 수 있어요.

In [13]:
# ---------------------------------------------------
# 스트리밍 출력 - 항목이 완성될 때마다 즉시 반환
# ---------------------------------------------------

print("=" * 60)
print("🔄 스트리밍 출력")
print("=" * 60)

# stream(): CommaSeparatedListOutputParser는 쉼표 구분 시마다 새 항목을 즉시 반환
# 각 chunk는 완성된 항목이 추가된 전체 리스트를 담고 있음
# TODO: chain.stream()을 사용하여 스트리밍 출력하세요

for chunk in chain.stream({"subject":"한국의 전통 음식"}):
    print(chunk, end="", flush=True)

🔄 스트리밍 출력
['김치']['비빔밥']['불고기']['잡채']['떡볶이']

## 실용 예제 1: 블로그 포스트 키워드 추출

블로그 포스트에서 SEO 키워드를 추출하는 시나리오입니다.


In [15]:
# ---------------------------------------------------
# 실용 예제 1: 블로그 포스트 SEO 키워드 추출
# ---------------------------------------------------

from langchain_core.prompts import ChatPromptTemplate

# 키워드 추출용 프롬프트
# TODO: ChatPromptTemplate.from_messages()로 프롬프트를 구성하세요
keyword_prompt = ChatPromptTemplate.from_messages([
  ("system", "당신은 SEO 전문가입니다. 주어진 텍스트에서 핵심 키워드를 추출하시오"),
  ("user", """
    다음 텍스트에서 SEO에 효과적인 키워드 5~7개를 추출하세요.
    텍스트:
    {text}

    {format_instructions}
  """),
])

# TODO: partial()로 형식 지침을 주입하세요
keyword_prompt = keyword_prompt.partial(format_instructions=output_parser.get_format_instructions())

# 체인 구성
# TODO: LCEL 파이프라인으로 체인을 구성하세요
keyword_chain = (
  keyword_prompt
  | model
  | output_parser
)

# 실행
blog_text = """
인공지능 기술이 빠르게 발전하면서 챗봇 개발이 더욱 쉬워졌습니다.
특히 LangChain 프레임워크는 대화형 AI 애플리케이션 구축을 간소화합니다.
Python을 사용하여 자연어 처리 파이프라인을 만들 수 있으며,
GPT 모델과의 통합도 매우 간단합니다.
"""

# TODO: keyword_chain.invoke()로 키워드를 추출하세요
res = keyword_chain.invoke({"text": blog_text})

print("\n" + "=" * 60)
print("🔍 추출된 SEO 키워드")
print("=" * 60)
# 반환된 list를 순회하며 출력
# TODO: 키워드 목록을 출력하세요
res



🔍 추출된 SEO 키워드


['인공지능', '챗봇 개발', 'LangChain', '대화형 AI', '자연어 처리', 'Python', 'GPT 모델']

## 실용 예제 2: 영화 추천 시스템

사용자의 선호도를 기반으로 영화를 추천하는 시나리오입니다.


In [ ]:
# 영화 추천 프롬프트
# TODO: PromptTemplate.from_template()으로 영화 추천 프롬프트를 구성하세요
#       입력 변수: genre, mood, recent_movie


# TODO: partial()로 형식 지침을 주입하세요


# 체인 구성
# TODO: LCEL 파이프라인으로 체인을 구성하세요


# 실행
# TODO: movie_chain.invoke()로 실행하세요


print("=" * 60)
print("🎬 추천 영화 목록")
print("=" * 60)
# TODO: 추천 목록을 출력하세요


## 실용 예제 3: 프로젝트 태스크 분해

큰 프로젝트를 작은 태스크로 분해하는 시나리오입니다.


In [16]:
# 태스크 분해 프롬프트
# TODO: ChatPromptTemplate.from_messages()로 프롬프트를 구성하세요
task_prompt = ChatPromptTemplate.from_messages([
  ("system", "당신은 프로젝트 관리 전문가입니다."),
  ("user", """
  다음 프로젝트를 5~8개의 구체적인 테스크로 분해하세요

  프로젝트" {project}

  각 테스크는 간결하게 작성해

  {format_instructions}
  """)
])

# TODO: partial()로 형식 지침을 주입하세요
task_prompt = task_prompt.partial(format_instructions=output_parser.get_format_instructions())

# 체인 구성
# TODO: LCEL 파이프라인으로 체인을 구성하세요
task_chain = (
  task_prompt
  | model
  | output_parser
)


# 실행
# TODO: task_chain.invoke()로 실행하세요
tasks = task_chain.invoke({"project": "블록체인을 만드는 프로젝트"})

print("\n" + "=" * 60)
print("📋 프로젝트 태스크 목록")
print("=" * 60)
print("프로젝트: 온라인 쇼핑몰 웹사이트 개발\n")
# TODO: 태스크 목록을 출력하세요

for i, task in enumerate(tasks, 1):
    print(f"✓ Task {i}: {task}")



📋 프로젝트 태스크 목록
프로젝트: 온라인 쇼핑몰 웹사이트 개발

✓ Task 1: 요구사항 분석
✓ Task 2: 시스템 아키텍처 설계
✓ Task 3: 스마트 계약 개발
✓ Task 4: 블록체인 네트워크 구축
✓ Task 5: 보안 및 테스트
✓ Task 6: 사용자 인터페이스 디자인
✓ Task 7: 문서화 및 교육
✓ Task 8: 배포 및 유지보수


## 실용 예제 4: 학습 주제 생성

특정 분야의 학습 로드맵을 생성하는 시나리오입니다.


In [ ]:
# 학습 주제 생성 프롬프트
# TODO: PromptTemplate.from_template()으로 학습 주제 생성 프롬프트를 구성하세요
#       입력 변수: level, subject


# TODO: partial()로 형식 지침을 주입하세요


# 체인 구성
# TODO: LCEL 파이프라인으로 체인을 구성하세요


# 실행
# TODO: learning_chain.invoke()로 실행하세요


print("=" * 60)
print("📚 학습 로드맵")
print("=" * 60)
print("분야: 데이터 과학 (초급)\n")
# TODO: 학습 주제 목록을 출력하세요


## 다양한 활용 예제

`CommaSeparatedListOutputParser`는 다양한 시나리오에서 활용할 수 있습니다.


In [ ]:
# 간단한 프롬프트로 다양한 목록 생성
# TODO: PromptTemplate.from_template()으로 간단한 프롬프트를 구성하세요


# TODO: partial()로 형식 지침을 주입하세요


# TODO: LCEL 파이프라인으로 체인을 구성하세요


# 1. 브레인스토밍
print("\n" + "=" * 60)
print("💡 브레인스토밍: 마케팅 아이디어")
print("=" * 60)
# TODO: simple_chain.invoke()로 마케팅 아이디어를 생성하고 출력하세요


# 2. 체크리스트 생성
print("\n" + "=" * 60)
print("✅ 체크리스트: 여행 준비물")
print("=" * 60)
# TODO: simple_chain.invoke()로 여행 준비물을 생성하고 출력하세요


# 3. 카테고리 분류
print("\n" + "=" * 60)
print("🏷️ 상품 카테고리")
print("=" * 60)
# TODO: simple_chain.invoke()로 상품 카테고리를 생성하고 출력하세요


## 핵심 요약

| 항목 | 설명 |
|------|------|
| **역할** | 쉼표 구분 텍스트 → Python `list` 변환 |
| **설정** | `CommaSeparatedListOutputParser()` 한 줄, 스키마 불필요 |
| **스트리밍** | 항목 완성 시마다 즉시 반환 |
| **제한** | 단순 문자열 리스트만 지원, 중첩 구조 불가 |

### 활용 시나리오별 비교

| 시나리오 | 추천 파서 | 이유 |
|----------|-----------|------|
| 키워드·해시태그 추출 | CommaSeparatedList | 단순 문자열 목록이면 충분 |
| 영화·책 추천 목록 | CommaSeparatedList | 제목만 필요할 때 |
| 추천 + 상세 정보 | JsonOutputParser | 제목·평점·설명 등 구조 필요 |
| 우선순위 분류 포함 | PydanticOutputParser | 타입 검증 + Enum 조합 |

## 다음 노트북 예고

**노트북 04 - 특수 타입 파싱** 에서는 `datetime`(날짜/시간)과 `Enum`(열거형) 타입을 Pydantic과 결합해서 파싱하는 방법을 배워요. 날짜 추출, 카테고리 분류, 감정 분석 등에 유용해요.

## 연습 문제

다음 연습 문제를 통해 `CommaSeparatedListOutputParser`의 활용법을 직접 실습해봅시다.

### 연습 1: 해시태그 생성기

`CommaSeparatedListOutputParser`를 사용하여 블로그 포스트 제목에서 **SNS 해시태그**를 자동 생성하는 체인을 만들어보세요.

**요구사항:**
- `ChatPromptTemplate`으로 시스템 메시지("당신은 SNS 마케팅 전문가입니다.")를 포함하는 프롬프트를 구성하세요
- 입력 변수: `post_title` (블로그 포스트 제목)
- 해시태그 7개를 생성하도록 프롬프트에 명시하세요
- 결과 리스트의 각 항목 앞에 `#`을 붙여 출력하세요
- 테스트 제목: "초보자를 위한 LangChain 입문 가이드: AI 앱 만들기"

In [ ]:
# ============================================================
# TODO: 해시태그 생성기 만들기
# 힌트: ChatPromptTemplate으로 시스템 메시지를 추가하고,
#       output_parser.get_format_instructions()를 partial()로 주입하세요
# 예상 결과: list 타입으로 해시태그 7개가 반환되고 앞에 # 붙여 출력
# ============================================================


### 연습 2: 스트리밍 퀴즈 생성기

`CommaSeparatedListOutputParser`와 `stream()` 메서드를 사용하여 특정 주제에 대한 **퀴즈 문제**를 스트리밍으로 생성하는 체인을 만들어보세요.

**요구사항:**
- `PromptTemplate`을 사용하세요
- 입력 변수: `subject` (과목), `count` (문제 수)
- `stream()` 메서드로 각 문제가 생성될 때마다 실시간 출력하세요
- 테스트: 과목 "한국사", 문제 수 5

In [ ]:
# ============================================================
# TODO: 스트리밍 퀴즈 생성기 만들기
# 힌트: PromptTemplate에 subject, count 변수를 정의하고
#       chain.stream()으로 실행하세요. 각 chunk는 리스트입니다
# 예상 결과: 각 퀴즈 문제가 완성될 때마다 실시간으로 출력됨
# ============================================================
